In [0]:
silver_df = spark.read.table("workspace.default.silver_merged_df")

unique_country_regions = [data[0] for data in silver_df.select('Country Name').distinct().collect()]

unique_indicators = [data[0] for data in silver_df.select('Indicator Name').distinct().collect()]

year_range = [str(year) for year in range(1960, 2021)]

In [0]:
cols = silver_df.columns

# Gold Layer - Avg over years
avg_sql = ", ".join(f"AVG(`{col}`) as `{'avg_'+col}`" for col in cols[2:])

gold_avg_df = spark.sql(f"""
                        SELECT `Indicator Name`, {avg_sql}
                        FROM silver_merged_df
                        GROUP BY `Indicator Name`
                        """)

gold_avg_df.show()

gold_avg_df.createOrReplaceTempView("gold_avg_df")
# gold_avg_df.write.option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("gold_avg_df")

# Min over years
min_sql = ", ".join(f"MIN(`{col}`) as `{'min_'+col}`" for col in cols[2:])

gold_min_df = spark.sql(f"""
                        SELECT `Indicator Name`, {min_sql}
                        FROM silver_merged_df
                        GROUP BY `Indicator Name`
                        """)

gold_min_df.show()

gold_min_df.createOrReplaceTempView("gold_min_df")
# gold_min_df.write.option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("gold_min_df")

# Max over years
max_sql = ", ".join(f"MAX(`{col}`) as `{'max_'+col}`" for col in cols[2:])

gold_max_df = spark.sql(f"""
                        SELECT `Indicator Name`, {max_sql}
                        FROM silver_merged_df
                        GROUP BY `Indicator Name`
                        """)

gold_max_df.show()

gold_max_df.createOrReplaceTempView("gold_max_df")
# gold_max_df.write.option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("gold_max_df")

# Median over years
median_sql = ", ".join(f"MEDIAN(`{col}`) as `{'median_'+col}`" for col in cols[2:])

gold_median_df = spark.sql(f"""
                        SELECT `Indicator Name`, {median_sql}
                        FROM silver_merged_df
                        GROUP BY `Indicator Name`
                        """)

gold_median_df.show()

gold_median_df.createOrReplaceTempView("gold_median_df")
# gold_median_df.write.option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("gold_median_df")

# Std. Deviation over years
std_sql = ", ".join(f"STDDEV(`{col}`) as `{'stddev_'+col}`" for col in cols[2:])

gold_std_df = spark.sql(f"""
                        SELECT `Indicator Name`, {std_sql}
                        FROM silver_merged_df
                        GROUP BY `Indicator Name`
                        """)

gold_std_df.show()

gold_std_df.createOrReplaceTempView("gold_std_df")

#gold_std_df.write.option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("gold_std_df")

In [0]:
# Yearwise Summary stats 

union_sql = """
SELECT 'min' as statistic, `Indicator Name`, {min_cols}
FROM gold_min_df
UNION ALL
SELECT 'max' as statistic, `Indicator Name`, {max_cols}
FROM gold_max_df
UNION ALL
SELECT 'median' as statistic, `Indicator Name`, {median_cols}
FROM gold_median_df
UNION ALL
SELECT 'avg' as statistic, `Indicator Name`, {avg_cols}
FROM gold_avg_df
UNION ALL
SELECT 'stddev' as statistic, `Indicator Name`, {stddev_cols}
FROM gold_std_df
ORDER BY `Indicator Name`, `statistic`
"""

summary_stats_df = spark.sql(union_sql.format(
    avg_cols=", ".join([f"`avg_{year}`" for year in cols[2:]]),
    min_cols=", ".join([f"`min_{year}`" for year in cols[2:]]),
    max_cols=", ".join([f"`max_{year}`" for year in cols[2:]]),
    median_cols=", ".join([f"`median_{year}`" for year in cols[2:]]),
    stddev_cols=", ".join([f"`stddev_{year}`" for year in cols[2:]])
))

summary_stats_df = summary_stats_df.toDF(*(['Statistic', 'Indicator Name'] + cols[2:]))

summary_stats_df.show(50, truncate=False)

summary_stats_df.write.option("delta.columnMapping.mode", "name").mode("overwrite").saveAsTable("gold_annual_statistics")